---

### Série 2

Usaremos a série `MSFT` — preço das ações da empresa Microsoft de 2020 a 2026.

---

## 1. Carregamento e visualização inicial

Os dados são obtidos diretamente do Yahoo Finance via pacote `quantmod`. A coluna `MSFT.Adjusted` contém o **preço ajustado de fechamento**, que desconta dividendos e splits — é a mais adequada para análise de série temporal.

### Sobre a frequência

Dados de ações são **diários**, com aproximadamente 252 pregões por ano. A escolha de `frequency` afeta apenas a decomposição e a detecção de sazonalidade:

| `frequency` | Sazonalidade detectada |
|-------------|------------------------|
| `5`         | Padrão semanal (5 pregões) |
| `252`       | Padrão anual (~252 pregões) |
| `1`         | Sem sazonalidade (mais honesto para preços) |

Usaremos **`frequency = 252`** para permitir a decomposição com perspectiva anual. Para a modelagem ARIMA, o valor de `frequency` não influencia o resultado — o que importa é a estrutura de dependência temporal.

In [ ]:
# Baixando os dados do Yahoo Finance
msft_raw <- getSymbols(
  "MSFT",
  src          = "yahoo",
  from         = "2020-01-01",
  to           = "2026-05-29",
  auto.assign  = FALSE
)

# Extraindo o preço ajustado e convertendo para ts
preco    <- as.numeric(msft_raw[, "MSFT.Adjusted"])
ts_msft  <- ts(preco, frequency = 252)   # ~252 pregões/ano

cat("Classe    :", class(ts_msft), "\n")
cat("Frequência:", frequency(ts_msft), "(pregões/ano)\n")
cat("N obs     :", length(ts_msft), "\n")
print(summary(ts_msft))

In [ ]:
# Visualização com datas reais no eixo x
datas <- index(msft_raw)

df_orig <- data.frame(
  data  = as.Date(datas),
  preco = preco
)

ggplot(df_orig, aes(x = data, y = preco)) +
  geom_line(color = "steelblue", linewidth = 0.7) +
  labs(
    title = "Preço Ajustado de Fechamento — Microsoft (MSFT)",
    x     = "Data",
    y     = "Preço (USD)"
  ) +
  theme_minimal(base_size = 13)

**Observações da visualização:**

- Há uma **tendência crescente** ao longo do período, com forte aceleração entre 2020 e 2021, e novamente a partir de 2023.
- A **variância não é constante**: a amplitude das oscilações cresce junto com o nível da série — comportamento típico de série **heterocedástica**, sugerindo que uma **transformação logarítmica** pode ser útil.
- Observa-se uma queda acentuada em 2022, possivelmente relacionada ao ciclo de alta de juros nos EUA.
- Visualmente, a série **não é estacionária** — tendência clara e variância crescente.

---
## 2. Decomposição da série

Com `frequency = 252`, a decomposição STL tentará isolar um padrão sazonal anual.
Para séries de preços de ações, esse componente sazonal tende a ser **muito fraco** — o que reforça que o grosso da variação está na tendência e no resíduo.

In [ ]:
# STL requer frequency >= 2; com 252 funciona bem
stl_msft <- stl(ts_msft, s.window = "periodic", robust = TRUE)

autoplot(stl_msft) +
  labs(title = "Decomposição STL — MSFT (frequência anual)") +
  theme_minimal(base_size = 12)

**O que observar na decomposição:**

- **Trend**: captura o movimento de longo prazo — crescimento com queda em 2022.
- **Seasonal**: se a amplitude for muito pequena em relação ao restante, confirma que **não há sazonalidade relevante**.
- **Remainder**: o resíduo — idealmente sem padrão. Se mostrar clusters de alta volatilidade, indica **heterocedasticidade** (comum em ações).

---
## 3. Teste de Estacionariedade

- **ADF**: $H_0$ = tem raiz unitária (não estacionária). $p < 0{,}05$ → estacionária.
- **KPSS**: $H_0$ = é estacionária. $p < 0{,}05$ → **não** estacionária.

In [ ]:
cat("=== Teste ADF — Série Original ===\n")
print(adf.test(ts_msft))

cat("\n=== Teste KPSS — Série Original ===\n")
print(kpss.test(ts_msft))

---
## 4. Transformações para Estacionarizar

Para séries financeiras com variância crescente, o procedimento padrão é:

1. **Transformação log** — estabiliza a variância (converte a série em *log-preços*)
2. **Diferenciação simples** — remove a tendência, convertendo log-preços em **log-retornos**

O log-retorno $r_t = \log(P_t) - \log(P_{t-1})$ é a métrica padrão em finanças para analisar variações percentuais de preço.

In [ ]:
# Log-preços
ts_log <- log(ts_msft)

cat("Diferenças simples sugeridas (ndiffs)  :", ndiffs(ts_log), "\n")
cat("Diferenças sazonais sugeridas (nsdiffs):", nsdiffs(ts_log), "\n")

In [ ]:
# Log-retornos = diff(log(preço))
ts_logret <- diff(ts_log)

# Visualização com datas
df_ret <- data.frame(
  data    = as.Date(datas[-1]),   # -1 pois diff reduz 1 obs
  logret  = as.numeric(ts_logret)
)

ggplot(df_ret, aes(x = data, y = logret)) +
  geom_line(color = "steelblue", linewidth = 0.4) +
  geom_hline(yintercept = 0, linetype = "dashed", color = "red") +
  labs(
    title = "Log-Retornos Diários — MSFT",
    x     = "Data",
    y     = "log(Pt / Pt-1)"
  ) +
  theme_minimal(base_size = 13)

In [ ]:
# Re-testando após transformação
cat("=== Teste ADF — Log-Retornos ===\n")
print(adf.test(ts_logret))

cat("\n=== Teste KPSS — Log-Retornos ===\n")
print(kpss.test(ts_logret))

---
## 5. Identificação do Modelo — ACF e PACF

Analisamos os correlogramas dos **log-retornos** (série estacionária) para identificar as ordens $p$ e $q$.

Para séries financeiras, é muito comum que os log-retornos se comportem como **ruído branco** — sem autocorrelação linear significativa. Isso levaria a um ARIMA(0,1,0) nos log-preços (equivalente a um *random walk*), que é o modelo de precificação eficiente de mercado.

In [ ]:
par(mfrow = c(1, 2), mar = c(4, 4, 3, 1))
acf(ts_logret,  lag.max = 30, main = "ACF — Log-Retornos")
pacf(ts_logret, lag.max = 30, main = "PACF — Log-Retornos")

---
## 6. Ajuste do Modelo ARIMA

Rodamos o `auto.arima()` na série de **log-preços** (não nos log-retornos). Isso porque o `auto.arima()` determina internamente o $d$, e queremos que o modelo final seja expresso como ARIMA$(p, d, q)$ na escala dos log-preços para facilitar a previsão.

In [ ]:
# auto.arima aplicado nos LOG-PREÇOS (série original transformada)
modelo_auto <- auto.arima(
  ts_log,
  stepwise      = FALSE,
  approximation = FALSE,
  trace         = TRUE
)

cat("\n=== Modelo selecionado automaticamente ===\n")
summary(modelo_auto)

In [ ]:
ord <- arimaorder(modelo_auto)
cat(sprintf("Modelo final: ARIMA(%d,%d,%d)\n", ord[1], ord[2], ord[3]))
cat(sprintf("AIC: %.2f | BIC: %.2f\n", AIC(modelo_auto), BIC(modelo_auto)))

---
## 7. Diagnóstico dos Resíduos

Verificamos se os resíduos do modelo se comportam como **ruído branco**:
- **Ljung-Box**: sem autocorrelação residual ($p > 0{,}05$)
- **Shapiro-Wilk**: normalidade dos resíduos
- **QQ-plot**: visualização da normalidade

> ⚠️ Em séries financeiras é comum que os resíduos não sejam normais (caudas pesadas). Isso não invalida o modelo ARIMA para previsão de nível, mas indica que modelos de volatilidade (GARCH) poderiam complementar a análise.

In [ ]:
checkresiduals(modelo_auto)

In [ ]:
res <- residuals(modelo_auto)

# Ljung-Box
fitdf_val <- sum(arimaorder(modelo_auto)[c(1, 3)])
lb <- Box.test(res, lag = 20, type = "Ljung-Box", fitdf = fitdf_val)
cat("=== Ljung-Box (lag=20) ===\n")
print(lb)
cat(if (lb$p.value > 0.05) "✅ Resíduos sem autocorrelação\n" else "⚠️  Autocorrelação residual presente\n")

# Shapiro-Wilk (amostra máxima de 5000)
res_sample <- as.numeric(res)
if (length(res_sample) > 5000) res_sample <- sample(res_sample, 5000)
sw <- shapiro.test(res_sample)
cat("\n=== Shapiro-Wilk (normalidade) ===\n")
print(sw)
cat(if (sw$p.value > 0.05) "✅ Resíduos normais\n" else "⚠️  Resíduos não normais (caudas pesadas — típico de séries financeiras)\n")

In [ ]:
qqnorm(res, main = "QQ-Plot dos Resíduos")
qqline(res, col = "red", lwd = 2)

---
## 8. Previsão

Geramos previsões para os próximos **30 pregões** (~6 semanas) na escala de log-preços e revertemos com `exp()` para a escala original de preço em USD.

In [ ]:
h <- 30  # 30 pregões à frente

prev_log <- forecast(modelo_auto, h = h, level = c(80, 95))

# Revertendo a transformação log → escala original
prev_orig        <- prev_log
prev_orig$mean   <- exp(prev_log$mean)
prev_orig$lower  <- exp(prev_log$lower)
prev_orig$upper  <- exp(prev_log$upper)
prev_orig$x      <- exp(prev_log$x)

autoplot(prev_orig) +
  labs(
    title    = "Previsão — MSFT (próximos 30 pregões)",
    subtitle = "Intervalos de confiança: 80% e 95%",
    x        = "Tempo (pregões desde jan/2020)",
    y        = "Preço Ajustado (USD)"
  ) +
  theme_minimal(base_size = 13)

In [ ]:
# Tabela com valores previstos
df_prev <- data.frame(
  Pregão    = seq_len(h),
  Previsão  = round(as.numeric(prev_orig$mean), 2),
  IC80_Lo   = round(as.numeric(prev_orig$lower[, 1]), 2),
  IC80_Hi   = round(as.numeric(prev_orig$upper[, 1]), 2),
  IC95_Lo   = round(as.numeric(prev_orig$lower[, 2]), 2),
  IC95_Hi   = round(as.numeric(prev_orig$upper[, 2]), 2)
)
print(df_prev)

---
## 9. Avaliação da Acurácia — Hold-out

Treinamos até o início de 2026 e avaliamos as previsões nos últimos ~100 pregões disponíveis.

In [ ]:
n      <- length(ts_log)
h_val  <- 100   # últimos 100 pregões como teste

treino <- ts(as.numeric(ts_log)[1:(n - h_val)],     frequency = 252)
teste  <- ts(as.numeric(ts_log)[(n - h_val + 1):n], frequency = 252)

ord_f       <- arimaorder(modelo_auto)
mod_treino  <- Arima(treino, order = c(ord_f[1], ord_f[2], ord_f[3]))
prev_treino <- forecast(mod_treino, h = h_val)

cat("=== Métricas de Acurácia (escala log) ===\n")
print(round(accuracy(prev_treino, teste), 6))

In [ ]:
autoplot(prev_treino) +
  autolayer(teste, series = "Valores Reais", color = "red", linewidth = 0.8) +
  labs(
    title = "Previsão vs. Valores Reais — Hold-out (últimos 100 pregões)",
    x     = "Tempo",
    y     = "log(Preço)",
    color = NULL
  ) +
  theme_minimal(base_size = 13)

---
## 📋 Resumo

| Etapa | Resultado |
|-------|-----------|
| Série | MSFT — preço ajustado diário (2020–2026) |
| Frequência | 252 pregões/ano |
| Comportamento | Tendência crescente, variância heterocedástica |
| Transformação | Log-preços (estabiliza variância) |
| Estacionariedade | Obtida com 1 diferença → log-retornos |
| Modelo | ARIMA(p,1,q) nos log-preços — ordens via `auto.arima` |
| Diagnóstico | Resíduos verificados via Ljung-Box, Shapiro-Wilk e QQ-plot |
| Horizonte | 30 pregões (~6 semanas) |

> **Nota:** Para séries financeiras, ARIMA captura apenas a dependência linear nos retornos. A volatilidade em clusters (períodos de alta/baixa oscilação) é melhor capturada por modelos **GARCH**, que seriam uma extensão natural deste trabalho.